# Phương pháp luận – Cơ sở thiết kế đặc trưng (Feature Design Rationale)

Quy trình xây dựng đặc trưng được thiết kế nhằm **chuyển dịch có hệ thống các giả thuyết về hành vi gian lận (domain-specific behavioral hypotheses) thành các tín hiệu có thể định lượng và đọc được bởi máy**. Thay vì tạo đặc trưng một cách tùy tiện, mỗi biến được xây dựng đều bắt nguồn từ những giả định rõ ràng về các mẫu hình giao dịch gian lận, đảm bảo rằng **không gian đặc trưng (feature space)** thu được vừa có thể giải thích được, vừa có khả năng phân biệt tốt.


## Nguyên tắc chỉ đạo

### 1. **Nguyên tắc Bảo toàn (Tính toàn vẹn Kế toán)**
   - **Giả thuyết**: Giao dịch hợp lệ phải tuân thủ nguyên tắc bảo toàn kế toán cơ bản:  
     `Số dư cũ − Số dư mới − Số tiền ≈ 0`.  
   - **Cách triển khai**:  
     - `balance_diff = oldbalanceOrg − newbalanceOrig − amount`  
     - `errorBalanceDest = newbalanceDest − oldbalanceDest − amount`  
   - **Cơ sở**: Các đặc trưng này mã hóa trực tiếp các vi phạm logic tài chính cơ bản, đóng vai trò là chỉ số chính cho thấy dấu hiệu giả mạo dữ liệu hoặc thanh toán gian lận.

### 2. **Độ lớn có ý nghĩa (Khuếch đại Tín hiệu)**
   - **Giả thuyết**: Độ lớn tuyệt đối của một vi phạm mang nhiều thông tin hơn là hướng của nó cho việc phát hiện bất thường.  
   - **Cách triển khai**:  
     - `abs_balance_diff = |balance_diff|`  
     - `abs_errorBalanceDest = |errorBalanceDest|`  
   - **Cơ sở**: Phép biến đổi này phù hợp với giả định của mô hình tuyến tính (Hồi quy Logistic) trong khi vẫn bảo toàn cường độ bất thường cho các mô hình dựa trên cây.

### 3. **Chuẩn hóa theo Ngữ cảnh (Bất thường Tương đối)**
   - **Giả thuyết**: Rủi ro của một giao dịch phải được xem xét tương đối so với bối cảnh tài chính của tài khoản, không phải giá trị tuyệt đối của nó.  
   - **Cách triển khai**:  
     - `amount_ratio_org = amount / (oldbalanceOrg + 1)`  
   - **Cơ sở**: Điều này ngăn chặn sự thiên lệch do quy mô — ví dụ: chuyển 1.000$ là đáng ngờ với tài khoản 1.500$ nhưng lại bình thường với tài khoản 1.000.000$.

### 4. **Phát hiện Trạng thái Biên (Hành vi trong Trường hợp Đặc biệt)**
   - **Giả thuyết**: Các tài khoản ở trạng thái số dư bằng không thể hiện phân phối hành vi khác biệt và dễ bị lợi dụng hơn.  
   - **Cách triển khai**:  
     - `is_empty_orig = (oldbalanceOrg == 0)`  
     - `is_empty_dest = (oldbalanceDest == 0)`  
   - **Cơ sở**: Các cờ nhị phân này cung cấp điểm phân tách rõ ràng cho thuật toán dạng cây và nắm bắt được các tương tác phi tuyến.

### 5. **Ổn định hóa Phân phối (Tính Mạnh mẽ Thống kê)**
   - **Giả thuyết**: Số tiền giao dịch thường có phân phối "đuôi dài"; phép biến đổi log giúp cải thiện độ ổn định và tốc độ hội tụ của mô hình.  
   - **Cách triển khai**:  
     - `log_amount = log(1 + amount)`  
   - **Cơ sở**: Giảm độ lệch, hạn chế ảnh hưởng của các giá trị cực đoan và đáp ứng tốt hơn giả định về phương sai không đổi (homoscedasticity) của các mô hình tuyến tính.

### 6. **Tín hiệu Heuristic Tổng hợp (Giám sát Yếu)**
   - **Giả thuyết**: Nhiều bất thường nhỏ xảy ra đồng thời sẽ làm tăng xác suất gian lận.  
   - **Cách triển khai**:  
     - `red_flag = (abs_balance_diff > ε) | (abs_errorBalanceDest > ε) | (amount_ratio_org > 0.9)`  
   - **Cơ sở**: Đặc trưng này đóng vai trò như một "tiên nghiệm" (prior) dẫn dắt mô hình hướng tới các vùng đáng ngờ mà không áp đặt các quy tắc cứng nhắc, hoạt động hiệu quả như một dạng của **học có giám sát yếu (weakly supervised learning)**.

### 7. **Rời rạc hóa Thời gian (Trích xuất Mẫu hình Chu kỳ)**
   - **Giả thuyết**: Các hành vi gian lận có thể thể hiện tính chu kỳ theo thời gian (ví dụ: theo giờ/ngày).  
   - **Cách triển khai**:  
     - `hour = step % 24`  
     - `day = step // 24`  
   - **Cơ sở**: Các đặc trưng thời gian tường minh cho phép mô hình nắm bắt tính thời vụ và ảnh hưởng của thời điểm trong ngày mà không cần dựa vào các mô hình tuần tự phức tạp.

## Tính phù hợp với Mô hình (Model-Aware Feature Suitability)

Tập đặc trưng được xây dựng có chủ ý để phục vụ **đồng thời cả mô hình tham số và phi tham số**:

- **Dành cho Hồi quy Logistic**: Các đặc trưng được chuẩn hóa (`StandardScaler`), biến đổi log và được thiết kế để xấp xỉ khả năng phân tách tuyến tính.
- **Dành cho Random Forest / XGBoost**: Các đặc trưng giữ lại độ lớn thô, cờ nhị phân và các tỷ lệ dễ tạo tương tác, cho phép cây quyết định khai thác các mối quan hệ phi tuyến và tương tác bậc cao.

## Loại bỏ các Định danh Không mang Thông tin

Các định danh có tính chất hồ sơ cao (`nameOrig`, `nameDest`) được loại bỏ rõ ràng để ngăn chặn **hiện tượng quá khớp do rò rỉ thông tin định danh (overfitting through identity leakage)**. Điều này đảm bảo mô hình có khả năng tổng quát hóa cho các thực thể chưa từng thấy thay vì chỉ ghi nhớ hành vi của các cá thể riêng lẻ.


In [1]:
# 0) IMPORTS
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

In [2]:
# 1) LOAD DATA

df = pd.read_csv(r"C:\Users\PC\PycharmProjects\PythonProject\ML\PROJECT\CHí BI=ình\data.csv")

# Basic sanity checks
assert "isFraud" in df.columns, "Target column 'isFraud' not found!"
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (325934, 23)
Columns: ['Unnamed: 0', 'step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'balance_diff', 'errorBalanceDest', 'abs_balance_diff', 'abs_errorBalanceDest', 'is_empty_orig', 'is_empty_dest', 'amount_ratio_org', 'log_amount', 'red_flag', 'hour', 'day']


In [3]:
# 2) FEATURE ENGINEERING (GỌN - GIẢI THÍCH ĐƯỢC)

def add_features(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()

    # --- 2.1) Balance logic (đã có sẵn trong dataset của bạn, nhưng bảo vệ nếu thiếu)
    # balance_diff = oldbalanceOrg - newbalanceOrig - amount
    if "balance_diff" not in df.columns:
        df["balance_diff"] = df["oldbalanceOrg"] - df["newbalanceOrig"] - df["amount"]

    # errorBalanceDest = newbalanceDest - oldbalanceDest - amount
    if "errorBalanceDest" not in df.columns:
        df["errorBalanceDest"] = df["newbalanceDest"] - df["oldbalanceDest"] - df["amount"]

    # --- 2.2) Độ lớn sai lệch (abs) giúp Logistic thấy rõ “mức độ vi phạm”
    df["abs_balance_diff"] = df["balance_diff"].abs()
    df["abs_errorBalanceDest"] = df["errorBalanceDest"].abs()

    # --- 2.3) Zero-balance flags (tài khoản trống: hành vi đáng nghi)
    df["is_empty_orig"] = (df["oldbalanceOrg"] == 0).astype(int)
    df["is_empty_dest"] = (df["oldbalanceDest"] == 0).astype(int)

    # --- 2.4) Amount ratio (giao dịch lớn so với số dư)
    # +1 để tránh chia 0
    df["amount_ratio_org"] = df["amount"] / (df["oldbalanceOrg"] + 1.0)

    # --- 2.5) Log amount (giảm lệch/phân phối đuôi dài; hữu ích cho Logistic)
    df["log_amount"] = np.log1p(df["amount"])

    # --- 2.6) Red flag tổng hợp (tín hiệu “có mùi”)
    # Ngưỡng nhỏ (eps) để tránh vấn đề float; không cần fancy
    eps = 1e-6
    df["red_flag"] = (
        (df["abs_balance_diff"] > eps) |
        (df["abs_errorBalanceDest"] > eps) |
        (df["amount_ratio_org"] > 0.90)
    ).astype(int)

    # --- 2.7) (Optional) Giữ hour/day nếu dataset đã có; nếu chưa có thì tạo từ step
    if "hour" not in df.columns and "step" in df.columns:
        df["hour"] = df["step"] % 24
    if "day" not in df.columns and "step" in df.columns:
        df["day"] = df["step"] // 24

    return df


df_fe = add_features(df)

In [4]:
df_fe

,Unnamed: 0,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,...,errorBalanceDest,abs_balance_diff,abs_errorBalanceDest,is_empty_orig,is_empty_dest,amount_ratio_org,log_amount,red_flag,hour,day
0,0,401,CASH_IN,290670.88,C1172770329,30613326.59,30903997.48,C1360884032,6252769.10,5962098.22,...,-5.813418e+05,581341.77,5.813418e+05,0,0,0.009495,12.579950,1,17,16
1,1,346,CASH_OUT,74969.02,C361253330,0.00,0.00,C1628643300,1510659.40,1585628.42,...,1.455192e-11,74969.02,1.455192e-11,1,0,74969.020000,11.224844,1,10,14
2,2,34,CASH_OUT,152359.15,C1296741848,0.00,0.00,C1234551068,519498.85,671857.99,...,-1.000000e-02,152359.15,1.000000e-02,1,0,152359.150000,11.934002,1,10,1
3,3,302,TRANSFER,553238.82,C765587136,0.00,0.00,C845383981,1278572.18,1831811.01,...,1.000000e-02,553238.82,1.000000e-02,1,0,553238.820000,13.223547,1,14,12
4,4,130,CASH_IN,432624.19,C852698921,18910.00,451534.19,C290472837,306529.18,0.00,...,-7.391534e+05,865248.38,7.391534e+05,0,0,22.876854,12.977627,1,10,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325929,325929,282,CASH_OUT,34345.29,C943859349,20380.00,0.00,C1322395030,0.00,34345.29,...,0.000000e+00,13965.29,0.000000e+00,0,1,1.685162,10.444249,1,18,11
325930,325930,379,PAYMENT,8234.77,C1729134111,272.00,0.00,M1758068394,0.00,0.00,...,-8.234770e+03,7962.77,8.234770e+03,0,1,30.163993,9.016242,1,19,15
325931,325931,235,CASH_OUT,13499.45,C297276080,0.00,0.00,C1955063491,84106.06,97605.52,...,1.000000e-02,13499.45,1.000000e-02,1,0,13499.450000,9.510478,1,19,9
325932,325932,369,CASH_OUT,236678.98,C2094429825,10018.00,0.00,C1407924212,0.00,236678.98,...,0.000000e+00,226660.98,0.000000e+00,0,1,23.623014,12.374464,1,9,15


In [5]:
# 3) Chuẩn bị X, y
# Drop high-cardinality ID-like columns (nameOrig/nameDest) để tránh overfit

DROP_COLS = [c for c in ["nameOrig", "nameDest"] if c in df_fe.columns]
print("Dropping:", DROP_COLS)

Dropping: ['nameOrig', 'nameDest']


In [6]:
df_fe.to_csv(r"C:\Users\PC\PycharmProjects\PythonProject\ML\PROJECT\CHí BI=ình\data.csv")

## Kết luận

Phương pháp luận xây dựng đặc trưng này là **hướng theo giả thuyết, không phải hướng theo dữ liệu (hypothesis-driven, not data-driven)**. Mỗi biến số là một đại diện toán học trực tiếp cho một giả định về hành vi gian lận tài chính, đảm bảo rằng mô hình thu được có thể giải thích được, mạnh mẽ và phù hợp với chuyên môn nghiệp vụ. Quy trình này hiệu quả trong việc thu hẹp khoảng cách giữa kiến thức nghiệp vụ định tính và học máy định lượng, biến các "cờ đỏ" trừu tượng thành các đặc trưng có thể tính toán và có khả năng phân biệt cao.